# Transcript Intelligence Analysis

This notebook documents the end-to-end analysis for the take-home assignment. It is written to be readable by product and engineering leadership as well as technical reviewers.

## 1. Assignment Objective

Build a pipeline that processes approximately 100 call transcripts across support, external/customer, and internal calls; categorize them into themes; analyze sentiment; and identify additional product/business insights. This run processed **100 transcripts**.

In [1]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'outputs').exists() and ROOT.name == 'notebooks':
    ROOT = ROOT.parent
processed = pd.read_csv(ROOT / 'outputs' / 'processed_transcripts.csv')
topics = pd.read_csv(ROOT / 'outputs' / 'topic_summary.csv')
sentiment = pd.read_csv(ROOT / 'outputs' / 'sentiment_summary.csv')
escalation = pd.read_csv(ROOT / 'outputs' / 'escalation_risks.csv')
features = pd.read_csv(ROOT / 'outputs' / 'feature_requests.csv')
churn = pd.read_csv(ROOT / 'outputs' / 'churn_risks.csv')
processed.head()

,transcript_id,source_folder,title,organizer_email,host,start_time,end_time,duration_minutes,email_domains,external_domains,...,lexical_sentiment_score,sentiment_key_phrases,provided_sentiment_norm,sentiment_score,sentiment_label,sentiment_strength,tfidf_cluster,cluster_keywords,theme,theme_description
0,01KQ03B0303900521BB089CA,data/raw/dataset/01KQ03B0303900521BB089CA,Detect Outage - Remediation Plan Review,megan.lawson@aegiscloud.com,megan.lawson@aegiscloud.com,2026-03-16T09:30:00.000Z,2026-03-16T10:05:12.000Z,35.2,"[""aegiscloud.com""]",[],...,0.304348,"[""good"", ""clear"", ""aligned"", ""solid"", ""outage""...",-0.3,-0.118696,neutral,0.119,6,"[""post"", ""outage"", ""customer"", ""mortem"", ""post...",Product bugs and technical reliability,"Reliability, outage, latency, backup, and tech..."
1,01KQ0C1280EDA4E70AAD7C35,data/raw/dataset/01KQ0C1280EDA4E70AAD7C35,Support Case #9279 - Summit Trust Billing Inquiry,sarah.chen@aegiscloud.com,sarah.chen@aegiscloud.com,2026-02-06T09:15:00.000Z,2026-02-06T09:38:48.000Z,23.8,"[""aegiscloud.com"", ""summittrust.com""]","[""summittrust.com""]",...,0.692308,"[""great"", ""good"", ""appreciate"", ""problem"", ""is...",0.4,0.487692,positive,0.488,4,"[""billing"", ""overage"", ""trust"", ""dispute"", ""su...","Renewal, pricing, and account risk","Commercial conversations where contract value,..."
2,01KQ0CAE7F064EC93F0540CA,data/raw/dataset/01KQ0CAE7F064EC93F0540CA,Weekly Engineering Standup,chris.lee@aegiscloud.com,chris.lee@aegiscloud.com,2026-02-16T10:00:00.000Z,2026-02-16T10:29:00.000Z,29.0,"[""aegiscloud.com""]",[],...,0.857143,"[""good"", ""confident"", ""risk""]",0.3,0.467143,positive,0.467,0,"[""sprint"", ""planning"", ""sprint planning"", ""tea...",Internal engineering planning,"Internal team coordination around roadmap, lau..."
3,01KQ0DFE299AC7A74E8022CA,data/raw/dataset/01KQ0DFE299AC7A74E8022CA,Aegis / Redwood Clinical - ISO 27001 Preparation,kevin.obrien@aegiscloud.com,kevin.obrien@aegiscloud.com,2026-04-26T13:30:00.000Z,2026-04-26T13:58:48.000Z,28.8,"[""aegiscloud.com"", ""redwoodclinical.com""]","[""redwoodclinical.com""]",...,1.000000,"[""clear"", ""good""]",0.9,0.930000,positive,0.930,3,"[""soc"", ""comply"", ""compliance"", ""hipaa"", ""repo...",Compliance and audit readiness,"Audit, evidence collection, control mapping, a..."
4,01KQ0F8AFF3DA34FD4580008,data/raw/dataset/01KQ0F8AFF3DA34FD4580008,Aegis / Cobalt Software - Q2 Planning,lisa.park@aegiscloud.com,lisa.park@aegiscloud.com,2026-04-11T14:15:00.000Z,2026-04-11T14:56:12.000Z,41.2,"[""aegiscloud.com"", ""cobaltsoftware.com""]","[""cobaltsoftware.com""]",...,0.300000,"[""appreciate"", ""great"", ""helpful"", ""outage"", ""...",0.2,0.230000,neutral,0.230,1,"[""renewal"", ""aegis"", ""outage"", ""pricing"", ""com...",Product bugs and technical reliability,"Reliability, outage, latency, backup, and tech..."


## 2. Dataset Overview

The raw data is a nested JSON folder structure. Each transcript folder contains meeting metadata, sentence-level transcript rows, speaker information, event rows, summary, action items, topics, sentiment, and key moments. Because no explicit `call_type` field exists, the pipeline infers support/external/internal using meeting title, content cues, and email domains.

In [2]:
processed[['transcript_id','title','call_type','customer_domain','speaker_count','word_count','theme','sentiment_label']].head(10)

,transcript_id,title,call_type,customer_domain,speaker_count,word_count,theme,sentiment_label
0,01KQ03B0303900521BB089CA,Detect Outage - Remediation Plan Review,internal,NaN,3,1324,Product bugs and technical reliability,neutral
1,01KQ0C1280EDA4E70AAD7C35,Support Case #9279 - Summit Trust Billing Inquiry,support,summittrust.com,2,1322,"Renewal, pricing, and account risk",positive
2,01KQ0CAE7F064EC93F0540CA,Weekly Engineering Standup,internal,NaN,4,1088,Internal engineering planning,positive
3,01KQ0DFE299AC7A74E8022CA,Aegis / Redwood Clinical - ISO 27001 Preparation,external,redwoodclinical.com,3,1420,Compliance and audit readiness,positive
4,01KQ0F8AFF3DA34FD4580008,Aegis / Cobalt Software - Q2 Planning,external,cobaltsoftware.com,2,1146,Product bugs and technical reliability,neutral
5,01KQ1267C6AA7D9B3125FEC8,SOC 2 Audit Preparation - Internal,internal,NaN,5,1532,Compliance and audit readiness,neutral
6,01KQ1A6B7E81B06F4A13B60D,Support Case #6977 - Brightpath Commerce Slow ...,support,brightpathcommerce.com,2,1161,Product bugs and technical reliability,neutral
7,01KQ1DC6CA536DE1B31ED8F5,Aegis / Atlas Precision - Contract Discussion,external,atlasprecision.com,3,1189,"Renewal, pricing, and account risk",positive
8,01KQ1DCC80852AE384C898C9,Aegis / Quantum Edge - Renewal Concerns,external,quantumedge.com,3,1536,"Renewal, pricing, and account risk",negative
9,01KQ1DE954A807A5D2653175,Comply v2 - Launch Readiness Review,support,NaN,2,1334,Compliance and audit readiness,positive


In [3]:
processed['call_type'].value_counts()

call_type
support     50
external    35
internal    15
Name: count, dtype: int64

![Call type breakdown](../outputs/charts/call_type_breakdown.png)

**Call-type inference result:** the corpus resolves to 50 support calls, 35 external/customer calls, and 15 internal calls. This matters because each stakeholder group reads transcript intelligence through a different lens: support needs issue triage, account teams need renewal risk, and engineering needs recurring technical patterns.

## 3. Data Cleaning and Assumptions

Cleaning removes obvious whitespace/timestamp noise while preserving the transcript language. Customer/account names are inferred from non-`aegiscloud.com` email domains. This is clearly documented as an assumption rather than treated as a perfect CRM account mapping.

## 4. Topic / Theme Categorization Approach

The pipeline uses a hybrid method: TF-IDF clustering for discovery, provided topic metadata for grounding, and business keyword rules for final human-readable themes. This is more practical than pure clustering, which often produces hard-to-explain labels, and safer than pure LLM labeling, which would require API access and validation.

In [4]:
topics[['theme','transcript_count','dominant_call_type','avg_sentiment_score','representative_keywords','business_importance']]

,theme,transcript_count,dominant_call_type,avg_sentiment_score,representative_keywords,business_importance
0,Product bugs and technical reliability,44,support,-0.126,"outage, incident, competitive, compliance, pip...",Turns repeated defects into prioritizable engi...
1,Compliance and audit readiness,25,external,0.661,"compliance, soc 2, hipaa, audit, reporting, co...",Connects product value to regulated buyer urge...
2,"Renewal, pricing, and account risk",16,support,0.347,"renewal, pricing, compliance, competitive, out...",Highlights revenue risk before the renewal mot...
3,Internal engineering planning,8,internal,0.523,"sprint planning, sprint, planning, retrospecti...",Reveals whether engineering execution is align...
4,Feature requests and product feedback,6,support,0.273,"compliance, feature, integration, api, workflo...",Converts customer language into roadmap eviden...
5,"Onboarding, adoption, and enablement",1,external,0.338,"onboarding, rollout, deployment planning, aler...",Shows where customers need enablement before l...


![Themes identified](../outputs/charts/theme_counts.png)

The taxonomy is intentionally business-readable. For example, **Product bugs and technical reliability** is more useful to leadership than a raw cluster label such as `cluster_2`, because it points directly to support burden, engineering ownership, and adoption risk.

### Example Transcript IDs and Excerpts by Theme

Each theme includes example transcript IDs and short excerpts so a reviewer can inspect whether the category is grounded in the underlying calls.

In [5]:
topic_examples = topics[['theme','example_transcripts','business_importance']].copy()
topic_examples

,theme,example_transcripts,business_importance
0,Product bugs and technical reliability,01KQ2D93184912F0147315E7: Julia Tran from Blac...,Turns repeated defects into prioritizable engi...
1,Compliance and audit readiness,01KQ445B1B5B890316C9E300: Raj facilitated a te...,Connects product value to regulated buyer urge...
2,"Renewal, pricing, and account risk",01KQ1DCC80852AE384C898C9: Maria Santos met wit...,Highlights revenue risk before the renewal mot...
3,Internal engineering planning,01KQ4EE836687F95080D92AB: The team held a spri...,Reveals whether engineering execution is align...
4,Feature requests and product feedback,01KQ56D878454E5783DC6D94: Damien Rowe from Str...,Converts customer language into roadmap eviden...
5,"Onboarding, adoption, and enablement",01KQ77F2867FC54E13A8C495: James Mitchell (AE) ...,Shows where customers need enablement before l...


## 5. Sentiment Analysis Approach

The source summaries include sentiment scores. The default implementation blends those structured scores with a transparent local lexical fallback. This keeps the pipeline runnable without paid API keys while making the scoring logic inspectable.

In [6]:
sentiment

,call_type,transcripts,avg_sentiment_score,negative_transcripts,neutral_transcripts,positive_transcripts,avg_word_count,interpretation
0,external,35,0.480,3,7,25,1571.486,External sentiment is a leading indicator for ...
1,internal,15,0.320,1,7,7,1242.867,Internal sentiment shows team confidence and u...
2,support,50,0.021,19,13,18,1354.240,Support sentiment reflects immediate product f...


![Sentiment distribution by call type](../outputs/charts/sentiment_distribution_by_call_type.png)

![Average sentiment by call type](../outputs/charts/average_sentiment_by_call_type.png)

**Interpretation:** support calls have the clearest negative signal. That is expected because customers usually contact support when something is broken, but the concentration around reliability/outage/SLA language means the signal should not stay inside support operations. Product and engineering should treat it as adoption and renewal risk.

## 6. Insight Findings

The most important trend is that support calls show the strongest negative sentiment, especially in reliability/outage/SLA-related themes. This should matter to support, product, and engineering because technical friction is showing up as customer trust risk, not just ticket volume.

![Sentiment by theme](../outputs/charts/sentiment_by_theme.png)

![Theme x call type sentiment heatmap](../outputs/charts/theme_calltype_sentiment_heatmap.png)

In [7]:
topics.sort_values('avg_sentiment_score')[['theme','transcript_count','dominant_call_type','avg_sentiment_score','business_importance']]

,theme,transcript_count,dominant_call_type,avg_sentiment_score,business_importance
0,Product bugs and technical reliability,44,support,-0.126,Turns repeated defects into prioritizable engi...
4,Feature requests and product feedback,6,support,0.273,Converts customer language into roadmap eviden...
5,"Onboarding, adoption, and enablement",1,external,0.338,Shows where customers need enablement before l...
2,"Renewal, pricing, and account risk",16,support,0.347,Highlights revenue risk before the renewal mot...
3,Internal engineering planning,8,internal,0.523,Reveals whether engineering execution is align...
1,Compliance and audit readiness,25,external,0.661,Connects product value to regulated buyer urge...


## 7. Additional Insights

The assignment asks, 'What else can you see?' The project implements escalation risk detection, feature request mining, churn/renewal risk signals, stakeholder-specific views, and product-engineering feedback-loop alignment. These are designed as product surfaces, not just analysis tables: each insight has a likely owner and suggested action.

In [8]:
escalation[['transcript_id','call_type','theme','risk_score','reason','suggested_owner']].head(10)

,transcript_id,call_type,theme,risk_score,reason,suggested_owner
0,01KQ2331EFD78BF3B1CAB747,support,Product bugs and technical reliability,9,"urgent, escalat, sla, outage, executive, critical",engineering + support
1,01KQ2D93184912F0147315E7,support,Product bugs and technical reliability,9,"escalat, sla, breach, outage, executive, critical",engineering + support
2,01KQ46A9DE0AECB006D897A0,support,Product bugs and technical reliability,9,"escalat, sla, outage, downtime, executive, cri...",engineering + support
3,01KQ38C4101D6774F2F02331,support,Product bugs and technical reliability,8,"escalat, sla, breach, outage, critical",engineering + support
4,01KQ6A5CEAAFFBE3B0099E75,support,Product bugs and technical reliability,8,"escalat, outage, downtime, critical, unresolved",engineering + support
5,01KQ94C2B8B9B4668F3D3491,support,Product bugs and technical reliability,8,"escalat, sla, breach, outage, critical",engineering + support
6,01KQ351E141926AB7CAB668D,external,Product bugs and technical reliability,7,"sla, breach, outage, executive",engineering + support
7,01KQ1DCC80852AE384C898C9,external,"Renewal, pricing, and account risk",7,"sla, breach, outage, postmortem",engineering + support
8,01KQ88B8EAEAA819C26CCBE7,support,Product bugs and technical reliability,7,"escalat, outage, executive, critical",engineering + support
9,01KQEDB92E33CF9945A7F71B,support,Product bugs and technical reliability,7,"escalat, sla, outage, critical",engineering + support


In [9]:
features.head(10)

,insight_type,requested_capability,frequency,impacted_call_types,example_excerpts,why_it_matters
0,Feature Request Mining,alerting and escalation,78,"{""internal"": 12, ""support"": 41, ""external"": 25}","01KQ03B0303900521BB089CA: Megan, Raj, and Bria...",Gives PMs roadmap evidence grounded in custome...
1,Feature Request Mining,export and reporting,77,"{""internal"": 11, ""external"": 33, ""support"": 33}","01KQ03B0303900521BB089CA: Megan, Raj, and Bria...",Gives PMs roadmap evidence grounded in custome...
2,Feature Request Mining,dashboarding and analytics,67,"{""internal"": 11, ""external"": 25, ""support"": 31}","01KQ03B0303900521BB089CA: Megan, Raj, and Bria...",Gives PMs roadmap evidence grounded in custome...
3,Feature Request Mining,onboarding and documentation,59,"{""internal"": 8, ""support"": 24, ""external"": 27}","01KQ03B0303900521BB089CA: Megan, Raj, and Bria...",Gives PMs roadmap evidence grounded in custome...
4,Feature Request Mining,integration and API reliability,52,"{""support"": 23, ""internal"": 7, ""external"": 22}","01KQ0C1280EDA4E70AAD7C35: Gregory Fisk, IT Dir...",Gives PMs roadmap evidence grounded in custome...
5,Feature Request Mining,granular restore and backup controls,33,"{""external"": 16, ""support"": 12, ""internal"": 5}",01KQ0DFE299AC7A74E8022CA: Daniel Okafor and so...,Gives PMs roadmap evidence grounded in custome...
6,Feature Request Mining,audit logs and data retention,32,"{""external"": 18, ""internal"": 4, ""support"": 10}",01KQ0DFE299AC7A74E8022CA: Daniel Okafor and so...,Gives PMs roadmap evidence grounded in custome...
7,Feature Request Mining,authentication and admin controls,31,"{""support"": 14, ""internal"": 5, ""external"": 12}","01KQ0C1280EDA4E70AAD7C35: Gregory Fisk, IT Dir...",Gives PMs roadmap evidence grounded in custome...
8,Feature Request Mining,automation and workflow,29,"{""external"": 12, ""internal"": 5, ""support"": 12}",01KQ0DFE299AC7A74E8022CA: Daniel Okafor and so...,Gives PMs roadmap evidence grounded in custome...
9,Feature Request Mining,billing and pricing visibility,24,"{""support"": 10, ""external"": 12, ""internal"": 2}","01KQ0C1280EDA4E70AAD7C35: Gregory Fisk, IT Dir...",Gives PMs roadmap evidence grounded in custome...


In [10]:
churn[['transcript_id','customer_or_account','risk_score','risk_terms','suggested_action']].head(10)

,transcript_id,customer_or_account,risk_score,risk_terms,suggested_action
0,01KQ1DCC80852AE384C898C9,Quantumedge,13,"renewal, competitor, pricing, budget, contract...","CSM/AM should review renewal plan, acknowledge..."
1,01KQ351E141926AB7CAB668D,Northstarpharma,8,"renewal, cancel, confidence, risk, sla","CSM/AM should review renewal plan, acknowledge..."
2,01KQ45C76AF3AEA6B3EA6697,Steelpointmfg,8,"renewal, competitor, pricing, contract, confid...","CSM/AM should review renewal plan, acknowledge..."
3,01KQ5A966832A146DA4B7D41,Summittrust,8,"renewal, competitor, pricing, trust, confidenc...","CSM/AM should review renewal plan, acknowledge..."
4,01KQ410FB7E434BA3F99EE7D,Axiomlabs.Dev,7,"renewal, pricing, contract, trust, procurement","CSM/AM should review renewal plan, acknowledge..."
5,01KQ8C5A044F54EE1774D53C,Helixdata,7,"competitor, confidence, risk, sla","CSM/AM should review renewal plan, acknowledge..."
6,01KQ56AA6B60801ABC01AB1C,Clearwatermed,6,budget,"CSM/AM should review renewal plan, acknowledge..."
7,01KQ4D71CDFB1045A2458216,Bridgeporthealth,6,"renewal, pricing, contract, trust","CSM/AM should review renewal plan, acknowledge..."
8,01KQ2D93184912F0147315E7,Blackridgeinvest,6,"competitor, trust, sla","CSM/AM should review renewal plan, acknowledge..."
9,01KQ1DC6CA536DE1B31ED8F5,Atlasprecision,6,"renewal, pricing, budget, contract","CSM/AM should review renewal plan, acknowledge..."


In [11]:
stakeholder = pd.read_csv(ROOT / 'outputs' / 'stakeholder_views.csv')
stakeholder

,stakeholder,primary_question,top_relevant_theme,relevant_transcripts,negative_transcripts,recommended_action
0,Support leader,Where are customers getting stuck and which is...,Product bugs and technical reliability,67,22,Create a weekly negative-theme and escalation-...
1,Product manager,What are customers asking for and which pain p...,Compliance and audit readiness,32,0,Validate top requests with PM taxonomy and att...
2,Sales / account manager,Which accounts show renewal risk or expansion ...,"Renewal, pricing, and account risk",44,4,Trigger proactive account plays for high-risk ...
3,Engineering lead,Which technical problems recur and how do they...,Product bugs and technical reliability,56,21,Tie incident and bug clusters to reliability r...


In [12]:
pe_gap = pd.read_csv(ROOT / 'outputs' / 'product_engineering_gap.csv')
pe_gap

,customer_facing_theme,customer_facing_count,internal_discussion_count,alignment_status,why_it_matters
0,Feature requests and product feedback,6,0,gap to investigate,Shows whether internal planning appears to mat...
1,"Onboarding, adoption, and enablement",1,0,gap to investigate,Shows whether internal planning appears to mat...
2,Product bugs and technical reliability,38,6,aligned,Shows whether internal planning appears to mat...
3,Compliance and audit readiness,22,3,aligned,Shows whether internal planning appears to mat...
4,"Renewal, pricing, and account risk",15,1,aligned,Shows whether internal planning appears to mat...
5,Internal engineering planning,3,5,aligned,Shows whether internal planning appears to mat...


## 8. Limitations

The taxonomy should be validated with stakeholders. Sentiment is context-sensitive, especially for incident calls. Account names are inferred from domains, not joined to CRM. Optional LLM labeling would add nuance, but it should be treated as an assisted workflow with human review.

## 9. Recommendations

Prioritize negative reliability themes, create an escalation-risk workflow, route feature requests into roadmap review, build stakeholder-specific dashboards, and validate the taxonomy with product/support/sales/engineering leadership.

## 10. Productionization Next Steps

Move from batch CSV outputs to a service-backed transcript intelligence layer: ingestion jobs, metadata store, embeddings index, validated classifiers, alerting rules, CRM/support integrations, and human feedback loops for taxonomy and model quality.